# Source Code Scanning

This notebook shows how mcpsafetywarden analyses the source code of a registered MCP server to find risks that are invisible at the protocol level: hardcoded secrets, dangerous imports, data flows from user-controlled parameters to shell or network sinks, and description mismatches that hint at deception.

**How it works**

When a GitHub URL is supplied, the scanner fetches the server's Python/TypeScript source files and runs up to seven analysis layers in sequence:

| Layer | Name | What it finds | Needs key? |
|---|---|---|---|
| 1 | entropy | Hardcoded secrets via Shannon entropy | No |
| 2 | ast | Dangerous imports, taint flows, sink calls | No |
| 3 | description_mismatch | Tool claims `read_only` but imports `subprocess` | No |
| 4 | rug_pull | Source hash changed since last scan | No |
| 5 | bandit | Python security linter findings | No |
| 6 | semgrep | Pattern-based security rules | No |
| 7 | llm | Deep semantic analysis of taint flows | API key |

**Three ways to run it**

1. `run_source_recon` directly - standalone, no server call needed
2. `security_scan_server` with `github_url` - source recon as part of the full MCPSafety+ pipeline
3. `mcpsafetywarden scan <server_id> --github-url <url>` - CLI

**Prerequisite:** `pip install mcpsafetywarden` and internet access to reach the GitHub API.

## 1. Setup

In [ ]:
import sys, json, os

from mcpsafetywarden.source_recon import run_source_recon
from mcpsafetywarden.server import register_server, security_scan_server, get_security_scan
from mcpsafetywarden import database as db

# Optional: set an LLM key to enable Layer 7 (deep semantic analysis).
# Layers 1-6 are fully offline and run without a key.
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'


def show(result):
    if isinstance(result, str):
        result = json.loads(result)
    print(json.dumps(result, indent=2))
    return result


SERVER_ID = 'source-scan-demo'

# GitHub URL to scan. Replace with the repo that implements your server.
# Here we use the official MCP Python SDK as a well-known example.
GITHUB_URL = 'https://github.com/modelcontextprotocol/python-sdk'

## 2. Standalone source recon

`run_source_recon` can be used without a registered server. Pass a minimal `server_config` dict and an optional list of known tools (used by Layer 3 to check description mismatches).

In [ ]:
# server_config only needs 'github_url' for a standalone scan.
server_config = {'github_url': GITHUB_URL}
tools = []  # Leave empty when not checking description mismatches

recon = await run_source_recon(
    server_id=SERVER_ID,
    server_config=server_config,
    tools=tools,
    github_url=GITHUB_URL,
    # llm_provider='anthropic',  # Uncomment to enable Layer 7
)

print(f"GitHub URL     : {recon['github_url']}")
print(f"Language       : {recon['language']}")
print(f"Files analyzed : {recon['files_analyzed']}")
print(f"Layers run     : {recon['layers_run']}")
print(f"Findings       : {len(recon['findings'])}")
print(f"Rug-pull alert : {recon['rug_pull_alert']}")

## 3. Interpret findings

Each finding has a `name`, `severity`, `finding` description, and optionally `file`, `line`, and `evidence`.

In [ ]:
findings = recon.get('findings', [])

if not findings:
    print('No findings - this repo looks clean across all offline layers.')
else:
    for f in findings:
        severity = f.get('severity', 'INFO')
        name     = f.get('name', 'unknown')
        finding  = f.get('finding', '')
        loc      = f.get('file', '')
        line     = f.get('line', '')
        loc_str  = f'{loc}:{line}' if loc and line else loc or ''
        print(f'[{severity:<8}] {name}')
        print(f'  {finding[:120]}')
        if loc_str:
            print(f'  {loc_str}')
        print()

## 4. Import capabilities and taint flows

**Import capabilities** list what the server can do at the OS level: `filesystem`, `network`, `subprocess`, `crypto`, `env_vars`, etc. These are derived from the actual `import` statements in the source.

**Taint flows** trace a path from a tool parameter (source) to a dangerous sink (e.g. `subprocess.run`, `os.system`, `requests.get`). Each flow is evidence of a potential injection vector.

In [ ]:
caps = recon.get('import_capabilities', [])
print(f'Import capabilities ({len(caps)}):')
for cap in caps:
    print(f'  - {cap}')

print()

sinks = recon.get('dangerous_sinks', [])
print(f'Dangerous sinks ({len(sinks)}):')
for sink in sinks[:5]:  # Show first 5
    print(f'  [{sink.get("sink_type", "?")}] {sink.get("function", "?")} in {sink.get("file", "?")}:{sink.get("line", "?")}')  # noqa

print()

flows = recon.get('taint_flows', [])
print(f'Taint flows ({len(flows)}):')
for flow in flows[:5]:  # Show first 5
    src  = flow.get('source_param', '?')
    sink = flow.get('sink', '?')
    fn   = flow.get('function', '?')
    print(f'  param `{src}` -> {sink} in {fn}')

## 5. Rug-pull detection

On the first scan the combined hash of all source files is stored in the database. Every subsequent scan recomputes the hash and compares it. If the files changed, `rug_pull_alert=True` is set and a finding is added with the old and new hash.

This catches the scenario where a server passes initial vetting, then quietly updates its code to add malicious behaviour.

In [ ]:
# After the first scan above, the hash is stored. Run it again.
recon2 = await run_source_recon(
    server_id=SERVER_ID,
    server_config=server_config,
    tools=tools,
    github_url=GITHUB_URL,
)

print(f'Rug-pull alert (second scan): {recon2["rug_pull_alert"]}')
print('No alert expected - source has not changed.\n')

# Inspect what is stored in the database.
stored = db.get_source_hash(SERVER_ID)
if stored:
    print(f'Stored hash      : {stored["files_hash"]}')
    print(f'First seen       : {stored["first_seen_at"]}')
    print(f'Last checked     : {stored["last_checked_at"]}')
    print(f'GitHub URL       : {stored["github_url"]}')

## 6. Source recon as part of the full scan pipeline

`security_scan_server` runs the MCPSafety+ five-stage pentest pipeline AND source recon when a `github_url` is available. The source findings are forwarded to the Planner and Auditor stages to generate more targeted attack hypotheses.

The `source_recon` key appears in the full scan result when source files were found.

In [ ]:
# Register the server first. We use the filesystem server as the live target;
# the github_url points to its source for static analysis.
# Replace both with your own server and repo in practice.
await register_server(
    server_id=SERVER_ID,
    transport='stdio',
    command='npx',
    args=['-y', '@modelcontextprotocol/server-filesystem', '/tmp'],
    github_url=GITHUB_URL,
    auto_inspect=True,
)

# Run the full scan. confirm_authorized=True is required - it confirms you
# own or are authorized to test this server. skip_web_research=True avoids
# leaking findings to external search engines.
scan_json = await security_scan_server(
    server_id=SERVER_ID,
    confirm_authorized=True,
    skip_web_research=True,
    background=False,
    github_url=GITHUB_URL,
)
scan = json.loads(scan_json)
print(f"Overall risk level : {scan.get('overall_risk_level')}")
print(f"Source recon present: {'source_recon' in scan}")

In [ ]:
# Inspect the source_recon section of the full scan.
source_recon = scan.get('source_recon', {})

if source_recon:
    print(f"GitHub URL     : {source_recon.get('github_url')}")
    print(f"Files analyzed : {source_recon.get('files_analyzed')}")
    print(f"Layers run     : {source_recon.get('layers_run')}")
    print(f"Findings       : {len(source_recon.get('findings', []))}")
    print(f"Rug-pull alert : {source_recon.get('rug_pull_alert')}")
    print(f"Capabilities   : {source_recon.get('import_capabilities')}")

    print('\nTool findings that include source evidence:')
    for finding in scan.get('tool_findings', []):
        if finding.get('source_evidence'):
            print(f"  [{finding['risk_level']}] {finding.get('finding', '')[:80]}")
else:
    print('source_recon not in scan result - no GitHub URL was resolved or source fetch timed out.')

## 7. Description mismatch detection

Layer 3 compares the server's advertised tool descriptions against what the source code actually imports and calls. A tool that claims to be read-only but imports `subprocess` or opens network connections is flagged as a potential deception.

To get mismatch findings, pass the registered tools to `run_source_recon`:

In [ ]:
tools_from_db = db.list_tools(SERVER_ID)

recon_with_tools = await run_source_recon(
    server_id=SERVER_ID,
    server_config={'github_url': GITHUB_URL},
    tools=tools_from_db,
    github_url=GITHUB_URL,
)

mismatch_findings = [
    f for f in recon_with_tools.get('findings', [])
    if 'mismatch' in f.get('name', '').lower() or 'capability' in f.get('name', '').lower()
]

print(f'Description mismatch findings: {len(mismatch_findings)}')
for f in mismatch_findings:
    print(f"  [{f.get('severity')}] {f.get('finding', '')[:120]}")

## CLI equivalent

```bash
# Scan with source code analysis (github_url auto-detected from PyPI metadata if possible)
mcpsafetywarden scan my-server --yes

# Explicit GitHub URL
mcpsafetywarden scan my-server --github-url https://github.com/owner/repo --yes

# Register with GitHub URL so it is stored and used on every future scan
mcpsafetywarden register my-server --transport streamable_http \
  --url https://mcp.example.com/mcp \
  --github-url https://github.com/owner/repo
```

**What `--yes` does:** it sets `confirm_authorized=True`, confirming you own or have written authorization to actively probe the server. Required for any scan that sends real payloads to the live server.